# Gradient Boosting - Beginner Guide

## What is Gradient Boosting?
Gradient Boosting builds **many small models (trees)**, one after another.

- Tree 1 makes a first guess.
- Tree 2 learns from the **mistakes** of Tree 1.
- Tree 3 learns from the mistakes that are still left.
- We add all the trees together. The final answer is much better.

Think of it like a team: each new person fixes the errors of the person before.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.datasets import load_breast_cancer

## Step 1: Make simple data
We make a curved line (sine wave) with a little noise.

In [ ]:
np.random.seed(42)
X = np.linspace(0, 10, 100).reshape(-1, 1)
y = np.sin(X).ravel() + np.random.normal(0, 0.2, 100)

plt.scatter(X, y, color="gray")
plt.title("Our data")
plt.show()

## Step 2: Build Gradient Boosting from scratch
Steps in the loop:
1. Start with a simple guess: the **average** of y.
2. Find the **residual** = actual value - predicted value (the mistake).
3. Train a small tree to predict the residual.
4. Add the tree's answer to our prediction (a small part, set by `learning_rate`).
5. Repeat.

In [ ]:
learning_rate = 0.1
n_trees = 50

pred = np.full(len(y), y.mean())   # first guess = average
errors = []

for i in range(n_trees):
    residual = y - pred                       # the mistakes
    tree = DecisionTreeRegressor(max_depth=2)
    tree.fit(X, residual)                     # learn the mistakes
    pred = pred + learning_rate * tree.predict(X)
    errors.append(mean_squared_error(y, pred))

print("Error after 1 tree  :", round(errors[0], 4))
print("Error after 50 trees:", round(errors[-1], 4))

In [ ]:
plt.scatter(X, y, color="gray", label="Data")
plt.plot(X, pred, color="red", label="Boosting prediction")
plt.legend()
plt.title("Prediction after 50 trees")
plt.show()

plt.plot(errors)
plt.xlabel("Number of trees")
plt.ylabel("Error (MSE)")
plt.title("Error goes down as we add trees")
plt.show()

## Step 3: Use scikit-learn (the easy way)
In real projects, we use the ready-made class. Same idea, less code.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = GradientBoostingRegressor(
    n_estimators=100,    # number of trees
    learning_rate=0.1,   # size of each step
    max_depth=2          # size of each tree
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("MSE:", round(mean_squared_error(y_test, y_pred), 4))
print("R2 :", round(r2_score(y_test, y_pred), 4))

## Step 4: Classification example
Now we predict a **category** (cancer: yes or no) using a real dataset.

In [ ]:
data = load_breast_cancer()
Xc, yc = data.data, data.target

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.2, random_state=42
)

clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
clf.fit(Xc_train, yc_train)

print("Accuracy:", round(accuracy_score(yc_test, clf.predict(Xc_test)), 4))

## Step 5: Which columns matter most?
Feature importance shows which columns helped the model the most.

In [ ]:
importances = clf.feature_importances_
top = np.argsort(importances)[-5:]

plt.barh(data.feature_names[top], importances[top])
plt.title("Top 5 important features")
plt.show()

## Step 6: Compare learning rates
Small learning rate = slow but careful. Big learning rate = fast but can overfit.

In [ ]:
for lr in [0.01, 0.1, 0.5, 1.0]:
    m = GradientBoostingRegressor(n_estimators=100, learning_rate=lr, max_depth=2, random_state=42)
    m.fit(X_train, y_train)
    mse = mean_squared_error(y_test, m.predict(X_test))
    print(f"learning_rate={lr:<5} Test MSE = {mse:.4f}")

## Summary

| Setting | Meaning |
|---|---|
| `n_estimators` | How many trees |
| `learning_rate` | How big each step is |
| `max_depth` | How big each tree is |

**Tips**
- Lower `learning_rate` + more trees is usually better.
- Too many deep trees can **overfit** (good on train, bad on test).
- Always check the error on **test data**.

## Practice
1. Change `max_depth` to 1, 3, 5. What happens to the error?
2. Try `n_estimators` = 10, 100, 500.
3. Try the model on your own CSV file with `pd.read_csv()`.